# LeetCode #55 — Jump Game: Greedy với "điểm xa nhất tới được"

**Mức độ:** Medium · **Chủ đề:** Array, Greedy, Dynamic Programming

Skeleton trên LeetCode:

```python
class Solution:
    def canJump(self, nums: List[int]) -> bool:
```

Mục tiêu của notebook:

1. Đi từ cách tự nhiên (**DP / đánh dấu tới được**) `O(n²)` tới **Greedy** `O(n)`.
2. Hiểu vì sao chỉ cần một biến `farthest`: tập vị trí tới được luôn là **một đoạn liền từ 0**.
3. Một góc nhìn Greedy khác: **dời đích về phía trái**.

# 1. Bài toán là gì?

Cho mảng `nums`, bạn đứng ở chỉ số `0`. `nums[i]` là **độ dài nhảy tối đa** từ vị trí `i`. Trả về `True` nếu tới được **chỉ số cuối cùng**.

**Constraints (theo đề):**

- `1 <= nums.length <= 10^4`
- `0 <= nums[i] <= 10^5`

---

## Ví dụ

```text
[2, 3, 1, 1, 4]  ->  True     (0 -> 1 -> 4)
[3, 2, 1, 0, 4]  ->  False    (mọi đường đều kẹt ở chỉ số 3, nums[3] = 0)
```

Ví dụ 2: từ 0 tới được 1, 2, 3; từ 1 tới được tối đa 3; từ 2 tới được tối đa 3; ở 3 thì `nums[3] = 0` — không đi tiếp được. Chỉ số 4 không bao giờ tới.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Thuật toán tham lam (greedy)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Thuật toán tham lam (greedy)

### Tham lam là gì?

Thuật toán tham lam xây lời giải bằng cách ở **mỗi bước chọn phương án tốt nhất trước mắt**, không bao giờ quay lại sửa. Nhanh và đơn giản — nhưng **không phải lúc nào cũng đúng**.

### Ví dụ đúng: đổi tiền với mệnh giá 1, 2, 5, 10

Luôn lấy tờ **lớn nhất** không vượt số tiền còn lại.

### Ví dụ sai: mệnh giá 1, 3, 4, cần 6

Tham lam: `4 + 1 + 1` (3 tờ). Tối ưu: `3 + 3` (2 tờ). Chọn tốt nhất trước mắt dẫn tới kết quả tệ hơn.

In [1]:
def doi_tien_tham_lam(menh_gia, so_tien):
    chon = []
    for m in sorted(menh_gia, reverse=True):
        while so_tien >= m:
            so_tien -= m
            chon.append(m)
    return chon

def doi_tien_toi_uu(menh_gia, so_tien):           # DP để đối chiếu
    INF = float("inf")
    dp = [0] + [INF] * so_tien
    for x in range(1, so_tien + 1):
        for m in menh_gia:
            if m <= x:
                dp[x] = min(dp[x], dp[x - m] + 1)
    return dp[so_tien]

print("mệnh giá [1,2,5,10], 18:", doi_tien_tham_lam([1, 2, 5, 10], 18), "| tối ưu:", doi_tien_toi_uu([1, 2, 5, 10], 18), "tờ")
print("mệnh giá [1,3,4], 6:", doi_tien_tham_lam([1, 3, 4], 6), "| tối ưu:", doi_tien_toi_uu([1, 3, 4], 6), "tờ")

mệnh giá [1,2,5,10], 18: [10, 5, 2, 1] | tối ưu: 4 tờ
mệnh giá [1,3,4], 6: [4, 1, 1] | tối ưu: 2 tờ


### Làm sao biết tham lam đúng?

Phải **chứng minh**. Hai kỹ thuật hay dùng:

1. **Lập luận trao đổi (exchange argument)**: giả sử có lời giải tối ưu khác lựa chọn tham lam ở bước đầu; chỉ ra có thể **đổi** nó sang lựa chọn tham lam mà không tệ đi.
2. **Bất biến "luôn dẫn đầu"**: sau mỗi bước, lời giải tham lam không kém bất kỳ lời giải nào khác.

Và luôn **thử phản ví dụ** bằng brute force trên dữ liệu nhỏ.

### Ví dụ kinh điển: chọn nhiều cuộc họp nhất không trùng giờ

Chọn cuộc họp **kết thúc sớm nhất** trước — để lại nhiều thời gian nhất cho phần còn lại (lập luận trao đổi: thay cuộc họp đầu của lời giải tối ưu bằng cuộc kết thúc sớm nhất vẫn không chồng lấn).

In [2]:
def nhieu_cuoc_hop(hop):
    chon, ket_thuc = [], float("-inf")
    for s, e in sorted(hop, key=lambda x: x[1]):     # theo giờ kết thúc
        if s >= ket_thuc:
            chon.append((s, e))
            ket_thuc = e
    return chon

hop_demo = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]
def theo_bat_dau(hop):                               # tiêu chí SAI để so sánh
    chon, ket_thuc = [], float("-inf")
    for s, e in sorted(hop):
        if s >= ket_thuc:
            chon.append((s, e)); ket_thuc = e
    return chon

print("kết thúc sớm nhất trước:", nhieu_cuoc_hop(hop_demo), "→", len(nhieu_cuoc_hop(hop_demo)), "cuộc")
print("bắt đầu sớm nhất trước (sai):", theo_bat_dau(hop_demo), "→", len(theo_bat_dau(hop_demo)), "cuộc")

kết thúc sớm nhất trước: [(1, 4), (5, 7), (8, 11), (12, 16)] → 4 cuộc
bắt đầu sớm nhất trước (sai): [(0, 6), (6, 10), (12, 16)] → 3 cuộc


### Lỗi hay gặp

- Tin tham lam đúng mà không chứng minh / không thử phản ví dụ.
- Chọn sai tiêu chí tham lam (bắt đầu sớm nhất, ngắn nhất… thay vì kết thúc sớm nhất).
- Quên sắp xếp theo tiêu chí trước khi duyệt.

## 2.2 Áp dụng vào bài này

**Tham lam**: duyệt từ trái, duy trì `farthest` = chỉ số xa nhất có thể tới. Nếu chỉ số hiện tại vượt `farthest` thì bị kẹt. Không cần thử mọi cách nhảy vì chỉ cần biết "tầm với" lớn nhất.

# 3. Cách 1: đánh dấu các vị trí tới được

`reach[i] = True` nếu tới được `i`. Bắt đầu `reach[0] = True`. Từ mỗi `i` tới được, đánh dấu `i+1 .. i+nums[i]`.

In [3]:
from typing import List

def can_jump_mark(nums: List[int]) -> bool:
    n = len(nums)
    reach = [False] * n
    reach[0] = True
    for i in range(n):
        if not reach[i]:
            continue
        for k in range(i + 1, min(n, i + nums[i] + 1)):
            reach[k] = True
    return reach[-1]

print(can_jump_mark([2, 3, 1, 1, 4]))
print(can_jump_mark([3, 2, 1, 0, 4]))

True
False


**Độ phức tạp:** `O(n · max(nums))` — với `nums[i]` tới `10^5`, có thể tới `10^9` thao tác. Quá chậm.

# 4. Quan sát: tập tới được là một đoạn liền

Nhìn mảng `reach` trong các ví dụ:

In [4]:
def reach_array(nums):
    n = len(nums)
    reach = [False] * n
    reach[0] = True
    for i in range(n):
        if reach[i]:
            for k in range(i + 1, min(n, i + nums[i] + 1)):
                reach[k] = True
    return reach

for nums in [[2, 3, 1, 1, 4], [3, 2, 1, 0, 4], [1, 0, 1, 0]]:
    print(nums, "->", ["T" if r else "." for r in reach_array(nums)])

[2, 3, 1, 1, 4] -> ['T', 'T', 'T', 'T', 'T']
[3, 2, 1, 0, 4] -> ['T', 'T', 'T', 'T', '.']
[1, 0, 1, 0] -> ['T', 'T', '.', '.']


Luôn có dạng `T T T ... T . . . .` — **một đoạn liền bắt đầu từ 0**, rồi toàn `.`.

**Vì sao?** Nếu tới được vị trí `p` (nhảy từ `i < p`), thì cũng tới được **mọi** vị trí giữa `i` và `p` (nhảy ngắn hơn luôn được phép). Lặp lại lập luận, mọi vị trí từ 0 tới `p` đều tới được.

> Vậy thay vì cả mảng `reach`, chỉ cần nhớ **một số**: `farthest` = vị trí xa nhất tới được. Mọi `i <= farthest` đều tới được.

# 5. Lời giải Greedy

```text
farthest = 0
for i in 0..n-1:
    if i > farthest: return False        # i nằm ngoài đoạn tới được -> kẹt
    farthest = max(farthest, i + nums[i])
    if farthest >= n - 1: return True     # dừng sớm
return True
```

In [5]:
def can_jump(nums: List[int]) -> bool:
    farthest = 0
    for i, x in enumerate(nums):
        if i > farthest:
            return False
        farthest = max(farthest, i + x)
        if farthest >= len(nums) - 1:
            return True
    return True

print(can_jump([2, 3, 1, 1, 4]))
print(can_jump([3, 2, 1, 0, 4]))
print(can_jump([0]))

True
False
True


# 6. Mô phỏng từng bước

`nums = [3, 2, 1, 0, 4]`

| i | nums[i] | i > farthest? | i + nums[i] | farthest sau |
|---|---|---|---|---|
| 0 | 3 | 0 > 0? không | 3 | 3 |
| 1 | 2 | 1 > 3? không | 3 | 3 |
| 2 | 1 | 2 > 3? không | 3 | 3 |
| 3 | 0 | 3 > 3? không | 3 | 3 |
| 4 | 4 | **4 > 3? có** | | → `False` |

In [6]:
def trace_can_jump(nums: List[int]) -> bool:
    farthest = 0
    print(f"nums = {nums}")
    for i, x in enumerate(nums):
        if i > farthest:
            print(f"i={i}: i > farthest = {farthest} -> không tới được i -> False")
            return False
        farthest = max(farthest, i + x)
        print(f"i={i} nums[i]={x}: farthest = {farthest}")
        if farthest >= len(nums) - 1:
            print(f"  farthest >= {len(nums) - 1} -> True")
            return True
    return True

_ = trace_can_jump([3, 2, 1, 0, 4])
print()
_ = trace_can_jump([2, 3, 1, 1, 4])

nums = [3, 2, 1, 0, 4]
i=0 nums[i]=3: farthest = 3
i=1 nums[i]=2: farthest = 3
i=2 nums[i]=1: farthest = 3
i=3 nums[i]=0: farthest = 3
i=4: i > farthest = 3 -> không tới được i -> False

nums = [2, 3, 1, 1, 4]
i=0 nums[i]=2: farthest = 2
i=1 nums[i]=3: farthest = 4
  farthest >= 4 -> True


# 7. Góc nhìn khác: dời đích về trái

Đi **từ phải sang trái**. Giữ `goal` = vị trí trái nhất **mà từ đó tới được đích**. Ban đầu `goal = n - 1`.

Với mỗi `i` từ `n - 2` về `0`: nếu `i + nums[i] >= goal` thì từ `i` nhảy tới được `goal` → `goal = i`.

Cuối cùng tới được đích khi và chỉ khi `goal == 0`.

In [7]:
def can_jump_backward(nums: List[int]) -> bool:
    goal = len(nums) - 1
    for i in range(len(nums) - 2, -1, -1):
        if i + nums[i] >= goal:
            goal = i
    return goal == 0

print(can_jump_backward([2, 3, 1, 1, 4]), can_jump_backward([3, 2, 1, 0, 4]))

True False


# 8. Phiên bản đúng format LeetCode

In [8]:
class Solution:
    def canJump(self, nums: List[int]) -> bool:
        farthest = 0
        for i, x in enumerate(nums):
            if i > farthest:
                return False
            farthest = max(farthest, i + x)
            if farthest >= len(nums) - 1:
                return True
        return True

In [9]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.canJump([2, 3, 1, 1, 4]), True)
check("Ví dụ 2", sol.canJump([3, 2, 1, 0, 4]), False)
check("Một phần tử", sol.canJump([0]), True)
check("Kẹt ngay đầu", sol.canJump([0, 1]), False)
check("Số 0 nhưng nhảy qua được", sol.canJump([2, 0, 0]), True)
check("Toàn số 1", sol.canJump([1, 1, 1, 1]), True)
check("Bước nhảy cực lớn", sol.canJump([100000, 0, 0, 0]), True)

import random
random.seed(55)
for _ in range(5000):
    nums = [random.randint(0, 3) for _ in range(random.randint(1, 15))]
    expected = can_jump_mark(nums)
    assert sol.canJump(nums) == expected, nums
    assert can_jump_backward(nums) == expected, nums
print("5000 test ngẫu nhiên: ba cách khớp nhau")

Ví dụ 1                    PASS  True
Ví dụ 2                    PASS  False
Một phần tử                PASS  True
Kẹt ngay đầu               PASS  False
Số 0 nhưng nhảy qua được   PASS  True
Toàn số 1                  PASS  True
Bước nhảy cực lớn          PASS  True


5000 test ngẫu nhiên: ba cách khớp nhau


# 9. Vì sao đúng? Độ phức tạp?

**Bất biến:** trước khi xét `i`, `farthest` là vị trí xa nhất tới được bằng cách chỉ nhảy từ các vị trí `< i`; và theo mục 4, **mọi** vị trí `<= farthest` đều tới được.

- Nếu `i <= farthest`: `i` tới được, nên mọi vị trí tới `i + nums[i]` cũng tới được → cập nhật `farthest`.
- Nếu `i > farthest`: không vị trí nào trước `i` nhảy tới được `i`, và cũng không tới được bất kỳ vị trí nào sau `i` (vì mọi vị trí tới được đều `<= farthest < i`). Kẹt → `False`.

```text
Time:  O(n)
Space: O(1)
```

| Cách | Thời gian | Bộ nhớ |
|---|---|---|
| Đánh dấu tới được | O(n · max(nums)) | O(n) |
| Greedy `farthest` | O(n) | O(1) |
| Greedy dời đích | O(n) | O(1) |

# 10. Những lỗi dễ mắc

## Lỗi 1: Coi mọi số `0` là "kẹt"

`[2, 0, 0]` vẫn `True`: từ chỉ số 0 nhảy thẳng 2 bước tới cuối, bỏ qua các số 0.

In [10]:
def wrong_any_zero(nums: List[int]) -> bool:
    return 0 not in nums[:-1]                      # SAI

print("Sai :", wrong_any_zero([2, 0, 0]), "| Đúng:", sol.canJump([2, 0, 0]))

Sai : False | Đúng: True


## Lỗi 2: Luôn nhảy xa nhất có thể từ vị trí hiện tại

Không phải lúc nào nhảy xa nhất cũng tốt: có thể rơi vào ô `0`.

In [11]:
def wrong_greedy_jump(nums: List[int]) -> bool:
    i = 0
    while i < len(nums) - 1:
        if nums[i] == 0:
            return False
        i = min(len(nums) - 1, i + nums[i])         # SAI: luôn nhảy tối đa
    return True

print("Sai :", wrong_greedy_jump([2, 3, 0, 0, 1]), "| Đúng:", sol.canJump([2, 3, 0, 0, 1]))

Sai : False | Đúng: True


Với `[2, 3, 0, 0, 1]`: nhảy xa nhất từ 0 là tới 2 (giá trị 0) → kẹt. Nhưng nhảy 1 bước tới chỉ số 1 (giá trị 3) rồi nhảy 3 bước tới cuối là được.

## Lỗi 3: Cập nhật `farthest` trước khi kiểm tra `i > farthest`

Nếu cập nhật trước, `farthest = max(farthest, i + nums[i]) >= i` luôn đúng → không bao giờ phát hiện kẹt.

## Lỗi 4: Quên trường hợp `n = 1`

Đã ở đích → `True`. Code trên trả `True` ngay ở vòng đầu (`farthest >= 0`).

# 11. Mẹo nhận diện pattern

- "**Có tới được không?**" trên mảng bước nhảy → Greedy `farthest`.
- "**Số bước ít nhất?**" → Greedy theo tầng (**#45 Jump Game II**).
- Nhảy **chính xác** `±nums[i]` → BFS / DFS (**#1306 Jump Game III**).
- Tập trạng thái tới được là **đoạn liền** → thay mảng boolean bằng **một biên**.

## Template

```python
farthest = 0
for i, x in enumerate(nums):
    if i > farthest: return False
    farthest = max(farthest, i + x)
return True
```

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Trace `[1, 2, 0, 1, 0, 3]` và `[2, 5, 0, 0]`.

## Bài 2 — Điền chỗ trống

In [12]:
def exercise_2_can_jump(nums: List[int]) -> bool:
    farthest = 0
    for i, x in enumerate(nums):
        # TODO 1: nếu i vượt farthest -> False
        # TODO 2: cập nhật farthest
        pass
    return True


# assert exercise_2_can_jump([2, 3, 1, 1, 4]) is True
# assert exercise_2_can_jump([3, 2, 1, 0, 4]) is False
# assert exercise_2_can_jump([0]) is True

## Bài 3 — Vị trí xa nhất tới được

Trả về **chỉ số xa nhất** có thể tới được (không nhất thiết là cuối mảng).

```python
farthest_reachable([3, 2, 1, 0, 4])  -> 3
farthest_reachable([2, 3, 1, 1, 4])  -> 4
farthest_reachable([0, 5])           -> 0
```

In [13]:
def exercise_3_farthest_reachable(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_3_farthest_reachable([3, 2, 1, 0, 4]) == 3
# assert exercise_3_farthest_reachable([2, 3, 1, 1, 4]) == 4
# assert exercise_3_farthest_reachable([0, 5]) == 0

## Bài 4 — Biến thể: Jump Game II (LeetCode #45)

Số lần nhảy **ít nhất** tới cuối (đề đảm bảo tới được).

```python
min_jumps([2, 3, 1, 1, 4]) -> 2
min_jumps([1, 1, 1, 1])    -> 3
```

In [14]:
def exercise_4_min_jumps(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_min_jumps([2, 3, 1, 1, 4]) == 2
# assert exercise_4_min_jumps([1, 1, 1, 1]) == 3
# assert exercise_4_min_jumps([0]) == 0

## Bài 5 — Mini challenge: Jump Game VII (LeetCode #1871)

Chuỗi nhị phân `s` (`s[0] == '0'`) và hai số `min_jump`, `max_jump`. Từ chỉ số `i` (với `s[i] == '0'`) được nhảy tới `j` nếu `i + min_jump <= j <= i + max_jump` và `s[j] == '0'`. Có tới được chỉ số cuối không? Yêu cầu `O(n)`.

```python
can_reach_vii("011010", 2, 3)   -> True
can_reach_vii("01101110", 2, 3) -> False
```

Gợi ý nhanh: `j` tới được nếu có **ít nhất một** `i` tới được trong cửa sổ `[j - max_jump, j - min_jump]`. Đếm số vị trí tới được trong cửa sổ bằng **cửa sổ trượt** (thêm/bớt một phần tử mỗi bước).

In [15]:
def exercise_5_can_reach_vii(s: str, min_jump: int, max_jump: int) -> bool:
    # TODO
    pass


# assert exercise_5_can_reach_vii("011010", 2, 3) is True
# assert exercise_5_can_reach_vii("01101110", 2, 3) is False
# assert exercise_5_can_reach_vii("00", 1, 1) is True

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_can_jump` để so sau khi tự làm.

**Bài 3**
- Mức 1: giống lời giải chính, nhưng khi `i > farthest` thì `break` thay vì trả `False`.
- Mức 2: kết quả là `min(farthest, n - 1)`.

**Bài 4**
- Mức 1: giữ `current_end` (biên tầng hiện tại) và `farthest`; khi `i == current_end` → tăng số bước.

**Bài 5**
- Mức 1: `reach = [False] * n`, `reach[0] = True`, `count = 0` (số vị trí tới được trong cửa sổ).
- Mức 2: với mỗi `j >= min_jump`: thêm `reach[j - min_jump]` vào `count`; nếu `j > max_jump` thì bớt `reach[j - max_jump - 1]`.
- Mức 3: `reach[j] = s[j] == '0' and count > 0`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [16]:
_ = trace_can_jump([1, 2, 0, 1, 0, 3])
print()
_ = trace_can_jump([2, 5, 0, 0])

nums = [1, 2, 0, 1, 0, 3]
i=0 nums[i]=1: farthest = 1
i=1 nums[i]=2: farthest = 3
i=2 nums[i]=0: farthest = 3
i=3 nums[i]=1: farthest = 4
i=4 nums[i]=0: farthest = 4
i=5: i > farthest = 4 -> không tới được i -> False

nums = [2, 5, 0, 0]
i=0 nums[i]=2: farthest = 2
i=1 nums[i]=5: farthest = 6
  farthest >= 3 -> True


## Đáp án Bài 2

In [17]:
def answer_2_can_jump(nums: List[int]) -> bool:
    farthest = 0
    for i, x in enumerate(nums):
        if i > farthest:
            return False
        farthest = max(farthest, i + x)
    return True

assert answer_2_can_jump([2, 3, 1, 1, 4]) is True
assert answer_2_can_jump([3, 2, 1, 0, 4]) is False
assert answer_2_can_jump([0]) is True
print("Bài 2 PASS")

Bài 2 PASS

## Đáp án Bài 3

In [18]:
def answer_3_farthest_reachable(nums: List[int]) -> int:
    farthest = 0
    for i, x in enumerate(nums):
        if i > farthest:
            break
        farthest = max(farthest, i + x)
    return min(farthest, len(nums) - 1)

assert answer_3_farthest_reachable([3, 2, 1, 0, 4]) == 3
assert answer_3_farthest_reachable([2, 3, 1, 1, 4]) == 4
assert answer_3_farthest_reachable([0, 5]) == 0
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [19]:
def answer_4_min_jumps(nums: List[int]) -> int:
    jumps = current_end = farthest = 0
    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        if i == current_end:
            jumps += 1
            current_end = farthest
    return jumps

assert answer_4_min_jumps([2, 3, 1, 1, 4]) == 2
assert answer_4_min_jumps([1, 1, 1, 1]) == 3
assert answer_4_min_jumps([0]) == 0
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [20]:
def answer_5_can_reach_vii(s: str, min_jump: int, max_jump: int) -> bool:
    n = len(s)
    reach = [False] * n
    reach[0] = True
    count = 0
    for j in range(1, n):
        if j >= min_jump and reach[j - min_jump]:
            count += 1
        if j > max_jump and reach[j - max_jump - 1]:
            count -= 1
        reach[j] = s[j] == "0" and count > 0
    return reach[-1]

assert answer_5_can_reach_vii("011010", 2, 3) is True
assert answer_5_can_reach_vii("01101110", 2, 3) is False
assert answer_5_can_reach_vii("00", 1, 1) is True

def brute_vii(s, lo, hi):
    n = len(s)
    reach = [False] * n
    reach[0] = True
    for i in range(n):
        if reach[i]:
            for j in range(i + lo, min(n, i + hi + 1)):
                if s[j] == "0":
                    reach[j] = True
    return reach[-1]

for _ in range(3000):
    n = random.randint(2, 12)
    s = "0" + "".join(random.choice("01") for _ in range(n - 1))
    lo = random.randint(1, n - 1)
    hi = random.randint(lo, n - 1)
    assert answer_5_can_reach_vii(s, lo, hi) == brute_vii(s, lo, hi), (s, lo, hi)
print("Bài 5 PASS (khớp brute force trên 3000 test)")

Bài 5 PASS (khớp brute force trên 3000 test)

# 15. Tổng kết

> Tập vị trí tới được luôn là **một đoạn liền từ 0** → chỉ cần một biến `farthest`. Duyệt `i`: nếu `i > farthest` thì kẹt; ngược lại mở rộng `farthest = max(farthest, i + nums[i])`.

---

## Checklist trước khi nộp

- [ ] Kiểm tra `i > farthest` **trước** khi cập nhật `farthest`?
- [ ] Không coi mọi số `0` là kẹt?
- [ ] Không dùng greedy "luôn nhảy xa nhất"?
- [ ] Đã thử `[0]`, `[0, 1]`, `[2, 0, 0]`?